In [2]:
from helper_functions import vita_df_to_text, text_to_vita_df, determine_candidates, call_chat_ai
import polars as pl
from openai import OpenAI
import json
import os
from dotenv import load_dotenv

# API configuration
BASE_URL = "https://chat-ai.academiccloud.de/v1"
#MODEL = "openai-gpt-oss-120b" # decent
#model = "apertus-70b-instruct-2509" # terrible
#model = "deepseek-r1-distill-llama-70b" # bad
MODEL = "gemma-4-31b-it" # decent? - doesn't expand what it shouldn't - does expand things that it shouldn't/hallucinates (expanded `A` without a `.`) when there is nothing to be expanded, but it makes sense to catch that case anyway
    # also did smart selection of expansion (used the expansion `(gratia ) expectativa` as just `expectativa`) - this could be both good and bad (doing smart adjustsments is good, but sometimes this could lead to errors)
    # Tru[d]perti' became 'Trudperti
#model = "glm-4.7" # decent?
#model = "mistral-large-3-675b-instruct-2512" # terrible? (didn't respect #5: Scope)
#model = "qwen3.5-397b-a17b" # terrible? - HUGE amount of reasoning and bad result (only so far tested on one example)
# mschonhardt/latin-normalizer terrible (for expanding) - leaves out stuff and halucinates (might be useful for actually normalising later though)
# andbue/byt5-base-latin-normalize useless for expanding
# dantedgp/latin-english-MT completely useless
# hathibelagal/llama-3.2-latin seemed promising, but my prompt (Please expand all abbreviations in the following text) for some reason leads to no output being generated.

STEP_SIZE = 10
MAX_BATCH_ATTEMPTS = 10
MAX_ROW_ATTEMPTS = 5
DEBUG = True

load_dotenv()  # load .env file into environment
api_key = os.environ["API_KEY"]

CLIENT = OpenAI(api_key=api_key, base_url=BASE_URL)

SYSTEM_PROMPT = """**Role:** You are a historian specializing in medieval church history with expert knowledge of the Latin abbreviations used in the papal registers.

**Task:** You will receive a Latin text in which some abbreviations are marked as `[[id|abbreviation]]`, together with a list of expansion candidates for each id. For every id, select the candidate that best fits the grammatical and semantic context of the surrounding text.

**Instructions:**

1. **Choices:** Choose exactly one candidate per id, from the given candidates only.

2. **Base forms:** Return the chosen candidate exactly as it is written in the candidate list. Do not inflect, alter, or extend it — the grammatical form is adjusted in a later processing step.

3. **Occurrences:** The same abbreviation can require different expansions at different places in the text; judge each occurrence in its own context. If you are unsure, prefer the expansion most commonly associated with that abbreviation in medieval Latin church documents.

4. **Output format:** Return only a single JSON object mapping every id to the chosen candidate, e.g. `{"1": "confirmatio", "2": "dominus"}`. Do not include explanations, commentary, or any other text.
"""

read data

In [3]:
rg = pl.read_csv("data/RG_header_sublemma_all.csv").select(["volume","nr_RG","nr_suffix","header_no_tags","regest_no_tags","id_RG_all"])
once_expanded = pl.read_csv("data/once_expanded.csv")

# both come out of extract_glossary.py (step 0); the Auflösungen in complex.csv
# are already stripped of the glossary's notes, so they can be used as candidates as they are
simple = pl.read_csv("data/simple.csv")
complex = pl.read_csv("data/complex.csv")

filter to include only a testing subset

In [4]:
ablass_ids = pl.read_csv("data/ablaesse.csv")

ablaesse = rg.join(ablass_ids,how="inner", on=["volume", "nr_RG"])
ablaesse_once_expanded = once_expanded.join(ablass_ids,how="inner", on=["volume", "nr_RG"])

**Multiple-choice reformat:** instead of having the model rewrite the whole text (which sometimes corrupted words it shouldn't touch, e.g. `Halberstad.` → `Halhalstad.`, silently expanded abbreviations without glossary entries, and required a fragile token diff for validation), the abbreviations with candidates are marked in the text as `[[id|abbreviation]]` and the model only returns a JSON object mapping each id to the chosen candidate. The substitution happens programmatically (`multiple_choice.py`), so the rest of the text cannot change and every choice is validated by a simple membership test against the candidate list. Chosen candidates are inserted in their base form; the inflection is left to the normalisation step.

In [5]:
from multiple_choice import find_candidate_occurrences, build_user_prompt, parse_choices, apply_choices

def expand_single(original_text: str, candidates: dict[str, list[str]], debug=False):

    occurrences = find_candidate_occurrences(original_text, candidates)
    if not occurrences:
        print("nothing to be expanded")
        return None

    user_prompt = build_user_prompt(original_text, occurrences, candidates)
    #print(user_prompt)

    choices, errors, dump = None, [], None
    for attempt in range(MAX_ROW_ATTEMPTS):
        dump = call_chat_ai(CLIENT, MODEL, SYSTEM_PROMPT, user_prompt)
        content = dump["choices"][0]["message"]["content"]
        choices, errors = parse_choices(content, occurrences, candidates)
        if choices is not None:
            break
        elif debug and attempt > 0:
            print(f"starting attempt #{attempt + 1}")

    if choices is None:
        # no parseable response after all attempts -> leave everything unexpanded
        choices = {}

    return {
        "text": apply_choices(original_text, occurrences, choices),
        "choices": [
            {"id": occ.id, "abbreviation": occ.matched, "choice": choices.get(occ.id)}
            for occ in occurrences
        ],
        "errors": errors,
        "user_prompt": user_prompt,
        "response": dump,
    }

In [6]:
volume = 5
#nr = 370
nr = 885

vita_df = ablaesse_once_expanded.filter((pl.col("volume") == volume) & (pl.col("nr_RG") == nr))
once_expanded_text = vita_df_to_text(vita_df)
candidates = determine_candidates(vita_df, simple, complex)

print(once_expanded_text)
print('-'*100 + '\n')
for key in candidates:
    print(f"{key}:")
    for value in candidates[key]:
        print(f"    {value}")

Brunswic Brunswicen. Halberstadensis et Hildesemensis diocc.
abbas et monasterium s. Egidii B. ordo sancti Benedicti Halberstadensis diocesis: de conserv. 30. iun. 1435 S 307 228vs.
par. ecclesia sancti Andree B. Hildesemensis diocesis Ludolpho Quirre archidiaconus in Stockem in eccl. Hildesemensis et rector d. parochialis ecclesia supplic. : de indulg. 30. iun. 1435 S 309 203r.
decanus, capitulum et singuli canonici collegiata ecclesia sancti Blasii B. unius de notabilioribus collegiata eccl. Saxonie Ottone, Wilhelmo et Hinrico Brunswicen. et Luneborgen. ducibus, patron. etiam supplic. : de incorp. parochialis ecclesia in Woden Weden Hildesemensis diocesis 4 marca argenti fabrice d. ecclesia sancti Blasii 2 marca argenti 13. october 1438 S 350 168vs.
proconsules, consules et universitas opidum B. Hildesemensis et Halberstadensis diocc.: de conserv. privilegium de non evocando eis a Sigismundo R.I. conc. et a Martino V. conf. 26. iun. 1436 S 323 235vs, exec.: abbas monasterium sancti P

In [7]:
result = expand_single(once_expanded_text, candidates, debug=True)
if result:
    twice_expanded_text = result["text"]
    print(twice_expanded_text)

Brunswic Brunswicen. Halberstadensis et Hildesemensis diocc.
abbas et monasterium sanctus Egidii B. ordo sancti Benedicti Halberstadensis diocesis: de conservare 30. iunius 1435 S 307 228vs.
parochialis ecclesia sancti Andree B. Hildesemensis diocesis Ludolpho Quirre archidiaconus in Stockem in eccl. Hildesemensis et rector dictus parochialis ecclesia supplicatio : de indulgentia 30. iunius 1435 S 309 203r.
decanus, capitulum et singuli canonici collegiata ecclesia sancti Blasii B. unius de notabilioribus collegiata eccl. Saxonie Ottone, Wilhelmo et Hinrico Brunswicen. et Luneborgen. ducibus, patronatus etiam supplicatio : de incorporatio parochialis ecclesia in Woden Weden Hildesemensis diocesis 4 marca argenti fabrice dictus ecclesia sancti Blasii 2 marca argenti 13. october 1438 S 350 168vs.
proconsules, consules et universitas opidum B. Hildesemensis et Halberstadensis diocc.: de conservare privilegium de non evocando eis a Sigismundo Romanorum Imperator concessio et a Martino V. c

In [8]:
for choice in result["choices"]:
    print(f"[[{choice['id']}]] {choice['abbreviation']} → {choice['choice']}")

print()
for error in result["errors"]:
    print(error["message"])

[[1]] s. → sanctus
[[2]] conserv. → conservare
[[3]] iun. → iunius
[[4]] par. → parochialis
[[5]] d. → dictus
[[6]] supplic. → supplicatio
[[7]] indulg. → indulgentia
[[8]] iun. → iunius
[[9]] patron. → patronatus
[[10]] supplic. → supplicatio
[[11]] incorp. → incorporatio
[[12]] d. → dictus
[[13]] conserv. → conservare
[[14]] R.I. → Romanorum Imperator
[[15]] conc. → concessio
[[16]] conf. → confirmatio
[[17]] iun. → iunius
[[18]] exec. → executio



# expanding a batch

In [9]:
results = []
ablaesse_twice_expanded = []
ids = ablaesse_once_expanded.select("volume", "nr_RG").unique().sort(by="*")

debug = True
print(f"expanding {len(ids)} vitae")

expanding 156 vitae


In [10]:
i = 0

for row in ids.iter_rows(named=True):

    vita_df = ablaesse_once_expanded.filter((pl.col("volume") == row["volume"]) & (pl.col("nr_RG") == row["nr_RG"]))
    once_expanded_text = vita_df_to_text(vita_df)
    candidates = determine_candidates(vita_df, simple, complex)

    if not candidates:
        print(f"skipped vita #{i} (volume {row["volume"]} - nr {row["nr_RG"]}) with no candidates to expand")
        continue

    result = expand_single(once_expanded_text, candidates, debug)
    if result is None:
        print(f"skipped vita #{i} (volume {row["volume"]} - nr {row["nr_RG"]}) with no occurrences of the candidate abbreviations")
        continue

    twice_expanded_text = result["text"]
    twice_expanded_df = text_to_vita_df(twice_expanded_text, row["volume"], row["nr_RG"])

    ablaesse_twice_expanded.append(twice_expanded_df)

    results.append({
        "candidates": candidates,
        "choices": result["choices"],
        "errors": result["errors"],
        "original_text": vita_df_to_text(ablaesse.filter((pl.col("volume") == row["volume"]) & (pl.col("nr_RG") == row["nr_RG"]))),
        "once_expanded_text": once_expanded_text,
        "twice_expanded_text": twice_expanded_text,
    })

    i += 1
    if debug:
        print(f"finished vita #{i-1}")
    elif i % 10 == 0:
        print(f"finished processing {i} vitas")
        #break

ablaesse_twice_expanded = pl.concat(ablaesse_twice_expanded)

finished vita #0
finished vita #1
finished vita #2
finished vita #3
finished vita #4
finished vita #5
finished vita #6
finished vita #7
finished vita #8
finished vita #9
finished vita #10
finished vita #11
finished vita #12
finished vita #13
finished vita #14
finished vita #15
finished vita #16
finished vita #17
finished vita #18
finished vita #19
finished vita #20
finished vita #21
finished vita #22
finished vita #23
finished vita #24
finished vita #25
finished vita #26
finished vita #27
finished vita #28
finished vita #29
finished vita #30
finished vita #31
finished vita #32
finished vita #33
finished vita #34
finished vita #35
finished vita #36
finished vita #37
finished vita #38
finished vita #39
finished vita #40
finished vita #41
finished vita #42
finished vita #43
finished vita #44
finished vita #45
finished vita #46
finished vita #47
finished vita #48
finished vita #49
finished vita #50
finished vita #51
finished vita #52
finished vita #53
finished vita #54
finished vita #55
fi

In [11]:
len(results), len(ablaesse_twice_expanded)

(156, 528)

In [12]:
with open("data/results.json", "w") as file:
    json.dump(results, file, indent=2)

ablaesse_twice_expanded.write_csv("data/twice_expanded.csv")

In [15]:
pattern = r"[a-zA-Z]{2,200}\."

# filtering for all volumes except 10, because we don't have any rules for volumes 10 yet and are dropping this one when expanding
counts_ablaesse = ablaesse.filter(pl.col("volume") < 10).with_columns(pl.col("header_no_tags").str.count_matches(pattern).sum().alias("countH"), pl.col("regest_no_tags").str.count_matches(pattern).sum().alias("countR"))
abbreviations_ablaesse = counts_ablaesse.row(0)[-2] + counts_ablaesse.row(0)[-1]

counts_once = ablaesse_once_expanded.filter(pl.col("volume") != 10).with_columns(pl.col("header_no_tags").str.count_matches(pattern).sum().alias("countH"), pl.col("regest_no_tags").str.count_matches(pattern).sum().alias("countR"))
abbreviations_once = counts_once.row(0)[-2] + counts_once.row(0)[-1]

counts_twice = ablaesse_twice_expanded.filter(pl.col("volume") != 10).with_columns(pl.col("header_no_tags").str.count_matches(pattern).sum().alias("countH"), pl.col("regest_no_tags").str.count_matches(pattern).sum().alias("countR"))
abbreviations_twice = counts_twice.row(0)[-2] + counts_twice.row(0)[-1]

print(f"abbreviations ablaesse: {abbreviations_ablaesse}")
print(f"abbreviations once: {abbreviations_once}")
print(f"abbreviations twice: {abbreviations_twice}")

abbreviations ablaesse: 4360
abbreviations once: 1628
abbreviations twice: 399
